# LLM Ichki Arxitekturasini Tahlil Qilish: GPT-2 → Qwen2.5-0.5B



## Vazifa maqsadi

Darsda GPT-2 (2019, OpenAI) modelining ichki tuzilishi `transformers` kutubxonasi
orqali o'rganilgan edi. Ushbu ishda o'sha kodlar **butunlay boshqa avlod
arxitekturasi** — `Qwen/Qwen2.5-0.5B` (2024, Alibaba) ga moslashtiriladi va
har bir farq **matematik jihatdan tekshirib** chiqiladi.

## Nima uchun aynan Qwen2.5-0.5B?

| Mezon | Izoh |
|---|---|
| Hajmi | ~0.5B parametr — CPU'da ham yuklanadi, GPU shart emas |
| Zamonaviyligi | 2024-yil, hozirgi SOTA retseptining barcha komponentlari bor |
| Farqlar soni | GPT-2 bilan **6 ta fundamental** arxitektura farqi mavjud |
| Ochiqligi | Gated emas, Apache 2.0 |

## Biz tekshiradigan 6 ta fundamental farq

1. **Position Encoding:** Learned Absolute (`wpe`) → **RoPE** (Rotary)
2. **Normalization:** `LayerNorm` → **`RMSNorm`**
3. **Attention:** Multi-Head (MHA) → **Grouped-Query Attention (GQA)**
4. **FFN:** `Linear→GELU→Linear` → **SwiGLU (3 ta matritsa)**
5. **QKV layout:** Fused `Conv1D(nf=2304)` → **alohida `q/k/v_proj` Linear**
6. **Regularizatsiya:** `Dropout(0.1)` → **Dropout umuman yo'q**


In [1]:
import sys, platform, torch, transformers

print("Python      :", sys.version.split())
print("Platform    :", platform.system(), platform.release())
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA mavjud :", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Ishlatiladigan device:", DEVICE)

# transformers versiyasi tekshiruvi
from packaging import version
assert version.parse(transformers.__version__) >= version.parse("4.40.0"), \
    "Qwen2 uchun transformers >= 4.40 kerak. Yangilang: pip install -U transformers"
print("\n Muhit tayyor")


Python      : ['3.13.5', '|', 'packaged', 'by', 'Anaconda,', 'Inc.', '|', '(main,', 'Jun', '12', '2025,', '16:37:03)', '[MSC', 'v.1929', '64', 'bit', '(AMD64)]']
Platform    : Windows 11
PyTorch     : 2.6.0+cu124
Transformers: 5.15.0
CUDA mavjud : True
Ishlatiladigan device: cuda

 Muhit tayyor


## 1. Hugging Face autentifikatsiyasi

Terminalda `hf auth login` qilingan bo'lsa, quyidagi cell `whoami` ni chiqaradi.
Login qilinmagan bo'lsa — `None` qaytadi, lekin Qwen2.5-0.5B gated bo'lmagani
uchun ish davom etaveradi (faqat rate-limit past bo'ladi).


## 2. Ikkala modelni yuklash (baseline + target)

Taqqoslash ilmiy bo'lishi uchun **GPT-2 ni ham yuklaymiz** — u bizning
"nazorat guruhi" (control group). Har bir Qwen xususiyatini GPT-2 ga
qarshi qo'yib ko'rsatamiz.

`AutoModel` — bu modelning **"tanasi"** (body), ya'ni `lm_head` siz versiyasi.
Chiqishi: `last_hidden_state`, shakli `[batch, seq_len, hidden_size]`.


In [3]:
from transformers import AutoModel, AutoTokenizer, AutoConfig

GPT2_ID = "gpt2"
QWEN_ID = "Qwen/Qwen2.5-0.5B"

print("GPT-2 yuklanmoqda...")
gpt2 = AutoModel.from_pretrained(GPT2_ID)
gpt2_tok = AutoTokenizer.from_pretrained(GPT2_ID)
gpt2.eval()
print(" GPT-2 tayyor:", type(gpt2).__name__)


GPT-2 yuklanmoqda...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

 GPT-2 tayyor: GPT2Model


In [4]:
print("Qwen2.5-0.5B yuklanmoqda (~1 GB, birinchi marta biroz vaqt oladi)...")

qwen = AutoModel.from_pretrained(
    QWEN_ID,
    dtype=torch.float32,      # tahlil uchun fp32 — aniqroq raqamlar
    attn_implementation="eager",  # attention weights'ni ko'rish uchun MAJBURIY
)
qwen_tok = AutoTokenizer.from_pretrained(QWEN_ID)
qwen.eval()

print(" Qwen tayyor:", type(qwen).__name__)


Qwen2.5-0.5B yuklanmoqda (~1 GB, birinchi marta biroz vaqt oladi)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

 Qwen tayyor: Qwen2Model


## 3. `print(model)` — arxitektura skeleti

Bu darsdagi asosiy diagnostika buyrug'i. PyTorch `nn.Module.__repr__()`
rekursiv ravishda barcha sub-modullarni chop etadi.


In [5]:
print("="*70)
print("GPT-2 (2019)")
print("="*70)
print(gpt2)


GPT-2 (2019)
GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [6]:
print("="*70)
print("QWEN2.5-0.5B (2024)")
print("="*70)
print(qwen)


QWEN2.5-0.5B (2024)
Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)


## 4. Ikki `print(model)` chiqishining qiyosiy tahlili

### GPT-2 chiqishi (dars notebook'idan)




### Darhol ko'zga tashlanadigan 8 ta farq

| # | GPT-2 | Qwen2.5 | Nima uchun o'zgargan? |
|---|---|---|---|
| 1 | `wte` + `wpe` | faqat `embed_tokens` | Pozitsiya endi embedding'da emas, **attention ichida** (RoPE) beriladi |
| 2 | vocab 50 257 | vocab **151 936** | Ko'p tilli BPE (xitoy, arab, kirill...) |
| 3 | `h` (12 blok) | `layers` (**24** blok) | Chuqurroq → kuchliroq reprezentatsiya |
| 4 | `LayerNorm` | `Qwen2RMSNorm` | ~15% tezroq, mean-centering keraksiz |
| 5 | `Conv1D(nf=2304)` fused | `q/k/v_proj` alohida | GQA'da K,V o'lchami Q dan farq qiladi → fuse qilib bo'lmaydi |
| 6 | K,V = 768 | K,V = **128** | **GQA** — KV cache 6× kichik |
| 7 | MLP: 2 matritsa + GELU | MLP: **3 matritsa** + SiLU | **SwiGLU** — gated FFN, sifatliroq |
| 8 | 4× `Dropout(0.1)` | Dropout **yo'q** | Katta korpuslarda 1 epoch → overfitting yo'q |

### Eng muhim savol: nega `k_proj` chiqishi 128?

GPT-2 da: 12 head × 64 dim = 768 → Q, K, V hammasi 768.  
Qwen'da: **14 ta Query head**, lekin atigi **2 ta Key/Value head**.




Bu **Grouped-Query Attention**. Inference paytida KV cache hajmi:
- GPT-2: har token, har qatlam uchun `2 × 768 = 1536` qiymat
- Qwen:  har token, har qatlam uchun `2 × 128 = 256` qiymat → **6× tejash**

Uzun kontekstda (32K token) bu xotira farqi hal qiluvchi ahamiyatga ega.


In [7]:
import pandas as pd

g, q = gpt2.config, qwen.config

rows = [
    ("Model klassi",        type(gpt2).__name__,                  type(qwen).__name__),
    ("Hidden size (d_model)", g.n_embd,                           q.hidden_size),
    ("Qatlamlar soni",      g.n_layer,                            q.num_hidden_layers),
    ("Attention head",      g.n_head,                             q.num_attention_heads),
    ("KV head",             g.n_head,                             q.num_key_value_heads),
    ("Head dim",            g.n_embd // g.n_head,                 q.hidden_size // q.num_attention_heads),
    ("FFN oraliq o'lcham",  4 * g.n_embd,                         q.intermediate_size),
    ("FFN kengayish nisbati", f"{4*g.n_embd/g.n_embd:.2f}x",       f"{q.intermediate_size/q.hidden_size:.2f}x"),
    ("Lug'at hajmi",        g.vocab_size,                         q.vocab_size),
    ("Maks. kontekst",      g.n_positions,                        q.max_position_embeddings),
    ("Aktivatsiya",         g.activation_function,                q.hidden_act),
    ("Norm turi",           "LayerNorm",                          "RMSNorm"),
    ("Norm eps",            g.layer_norm_epsilon,                 q.rms_norm_eps),
    ("Pozitsiya kodlash",   "Learned Absolute",                   f"RoPE (theta={getattr(q,'rope_theta','n/a')})"),
    ("Dropout",             g.resid_pdrop,                        0.0),
    ("Embedding tied?",     "True",                               str(getattr(q, "tie_word_embeddings", None))),
]

df = pd.DataFrame(rows, columns=["Parametr", "GPT-2 small", "Qwen2.5-0.5B"])
pd.set_option("display.max_colwidth", 40)
display(df)


,Parametr,GPT-2 small,Qwen2.5-0.5B
0,Model klassi,GPT2Model,Qwen2Model
1,Hidden size (d_model),768,896
2,Qatlamlar soni,12,24
3,Attention head,12,14
4,KV head,12,2
5,Head dim,64,64
6,FFN oraliq o'lcham,3072,4864
7,FFN kengayish nisbati,4.00x,5.43x
8,Lug'at hajmi,50257,151936
9,Maks. kontekst,1024,32768


### Config tahlili — 4 ta muhim kuzatuv

**1. `intermediate_size = 4864`, ya'ni 5.43×, 4× emas**

GPT-2 da klassik qoida: `d_ff = 4 × d_model`. Qwen'da SwiGLU 3 ta matritsa
ishlatgani uchun, **bir xil parametr byudjetini saqlab qolish** maqsadida
kengayish nisbati kamaytirilgan:
- GPT-2 FFN parametri: `2 × d × 4d = 8d²`
- SwiGLU FFN parametri: `3 × d × 5.43d = 16.3d²`

Qwen boshqacha muvozanat tanlagan — FFN'ga ko'proq sarflab, attention'dan
(GQA hisobiga) tejagan.

**2. `rope_theta = 1_000_000`**

Original RoPE'da θ = 10 000 edi. Qwen uni **100× oshirgan**, chunki bu past
chastotali komponentlarni "sekinlashtiradi" va model 32K tokengacha
uzun kontekstni ushlay oladi.

**3. `max_position_embeddings = 32768` (GPT-2: 1024)**

32× uzunroq kontekst. Buni **faqat RoPE** imkon beradi: `wpe` bo'lganida
1024×768 jadvalni 32768×768 ga kengaytirish uchun **qayta o'qitish** kerak bo'lardi.
RoPE esa parametrsiz funksiya — extrapolyatsiya qilinadi.

**4. `tie_word_embeddings = True`**

0.5B modelda `embed_tokens` va `lm_head` **bitta va o'sha** matritsa.
151936 × 896 = 136 M parametr — bu modelning ~27%! Tie qilmasa 272 M bo'lardi.
Kichik modellarda bu majburiy optimizatsiya.


In [8]:
def count_params(model, name):
    total = sum(p.numel() for p in model.parameters())
    emb   = sum(p.numel() for n, p in model.named_parameters()
                if "embed" in n or n.startswith("wte") or n.startswith("wpe"))
    print(f"{name}")
    print(f"  Jami parametr        : {total:,}  (~{total/1e6:.1f}M)")
    print(f"  Embedding parametri  : {emb:,}  ({100*emb/total:.1f}%)")
    print(f"  Non-embedding        : {total-emb:,}  (~{(total-emb)/1e6:.1f}M)")
    print(f"  fp32 xotira          : {total*4/1e9:.2f} GB")
    print(f"  bf16 xotira          : {total*2/1e9:.2f} GB")
    print()
    return total

t_g = count_params(gpt2, "GPT-2 small")
t_q = count_params(qwen, "Qwen2.5-0.5B")
print(f"Qwen / GPT-2 nisbati: {t_q/t_g:.2f}x")


GPT-2 small
  Jami parametr        : 124,439,808  (~124.4M)
  Embedding parametri  : 39,383,808  (31.6%)
  Non-embedding        : 85,056,000  (~85.1M)
  fp32 xotira          : 0.50 GB
  bf16 xotira          : 0.25 GB

Qwen2.5-0.5B
  Jami parametr        : 494,032,768  (~494.0M)
  Embedding parametri  : 136,134,656  (27.6%)
  Non-embedding        : 357,898,112  (~357.9M)
  fp32 xotira          : 1.98 GB
  bf16 xotira          : 0.99 GB

Qwen / GPT-2 nisbati: 3.97x


In [10]:
layer = qwen.layers[0]          # ← [0] MUHIM: birinchi decoder layer

def n(m): return sum(p.numel() for p in m.parameters())

attn_p = n(layer.self_attn)
mlp_p  = n(layer.mlp)
norm_p  = n(layer.input_layernorm) + n(layer.post_attention_layernorm)
tot     = attn_p + mlp_p + norm_p

print("QWEN2 BITTA DECODER LAYER PARAMETR TAQSIMOTI")
print("-"*50)
print(f"  q_proj    : {n(layer.self_attn.q_proj):>10,}")
print(f"  k_proj    : {n(layer.self_attn.k_proj):>10,}")
print(f"  v_proj    : {n(layer.self_attn.v_proj):>10,}")
print(f"  o_proj    : {n(layer.self_attn.o_proj):>10,}")
print(f"  ATTENTION : {attn_p:>10,}  ({100*attn_p/tot:.1f}%)")
print("-"*50)
print(f"  gate_proj : {n(layer.mlp.gate_proj):>10,}")
print(f"  up_proj   : {n(layer.mlp.up_proj):>10,}")
print(f"  down_proj : {n(layer.mlp.down_proj):>10,}")
print(f"  MLP       : {mlp_p:>10,}  ({100*mlp_p/tot:.1f}%)")
print("-"*50)
print(f"  RMSNorm×2 : {norm_p:>10,}  ({100*norm_p/tot:.1f}%)")
print(f"  JAMI      : {tot:>10,}")

L = qwen.config.num_hidden_layers
print(f"\n  {L} qatlam : {tot*L:,}  (~{tot*L/1e6:.0f}M)")


QWEN2 BITTA DECODER LAYER PARAMETR TAQSIMOTI
--------------------------------------------------
  q_proj    :    803,712
  k_proj    :    114,816
  v_proj    :    114,816
  o_proj    :    802,816
  ATTENTION :  1,836,160  (12.3%)
--------------------------------------------------
  gate_proj :  4,358,144
  up_proj   :  4,358,144
  down_proj :  4,358,144
  MLP       : 13,074,432  (87.7%)
--------------------------------------------------
  RMSNorm×2 :      1,792  (0.0%)
  JAMI      : 14,912,384

  24 qatlam : 357,897,216  (~358M)


### Kuzatuv: MLP parametrlarning ~88% ini egallaydi

Bu zamonaviy LLM'larning umumiy xususiyati. Attention "aqli" (qaysi tokenlar
bir-biriga bog'liq) uchun javob bersa, MLP **bilim omborchisi** (knowledge store)
vazifasini bajaradi. Mexanistik interpretabellik tadqiqotlari (Geva et al., 2021)
MLP qatlamlarini "key-value memory" sifatida talqin qiladi.

**Amaliy xulosa:** Agar modelni siqmoqchi bo'lsangiz (pruning/quantization),
eng katta yutuq MLP'dan keladi, attention'dan emas.


## 5. Embedding qatlami: `wte` → `embed_tokens`

Dars notebook'idagi kodlar:
```python
print(model.wte)                 # Embedding(50257, 768)
print(model.wpe)                 # Embedding(1024, 768)
print(model.wte.weight.shape)    # torch.Size([50257, 768])
print(model.wte.weight)


In [14]:
print("GPT-2:")
print("  wte:", gpt2.wte)
print("  wpe:", gpt2.wpe)
print("  wte.weight.shape:", tuple(gpt2.wte.weight.shape))

print("\nQwen2.5:")
print("  embed_tokens:", qwen.embed_tokens)
print("  embed_tokens.weight.shape:", tuple(qwen.embed_tokens.weight.shape))
print("  wpe mavjudmi? :", hasattr(qwen, "wpe"))

print("\n  Qwen'dagi top-level modullar:")
for name, _ in qwen.named_children():
    print("    -", name)

GPT-2:
  wte: Embedding(50257, 768)
  wpe: Embedding(1024, 768)
  wte.weight.shape: (50257, 768)

Qwen2.5:
  embed_tokens: Embedding(151936, 896)
  embed_tokens.weight.shape: (151936, 896)
  wpe mavjudmi? : False

  Qwen'dagi top-level modullar:
    - embed_tokens
    - layers
    - norm
    - rotary_emb


In [15]:
import torch

def emb_stats(w, name):
    w = w.detach().float()
    norms = w.norm(dim=1)
    print(f"{name}")
    print(f"  shape      : {tuple(w.shape)}")
    print(f"  mean       : {w.mean():+.5f}")
    print(f"  std        : {w.std():.5f}")
    print(f"  min / max  : {w.min():+.3f} / {w.max():+.3f}")
    print(f"  L2 norm    : o'rtacha={norms.mean():.3f}, std={norms.std():.3f}")
    print()

emb_stats(gpt2.wte.weight, "GPT-2 wte")
emb_stats(qwen.embed_tokens.weight, "Qwen2 embed_tokens")


GPT-2 wte
  shape      : (50257, 768)
  mean       : +0.00038
  std        : 0.14370
  min / max  : -1.270 / +1.785
  L2 norm    : o'rtacha=3.959, std=0.434

Qwen2 embed_tokens
  shape      : (151936, 896)
  mean       : +0.00005
  std        : 0.01561
  min / max  : -0.205 / +0.166
  L2 norm    : o'rtacha=0.463, std=0.061



###  Muhim kuzatuv: Qwen embeddinglari GPT-2 dan ~10× KICHIKROQ std ga ega

Sabab — **`tie_word_embeddings=True`**.

GPT-2 da `wte` faqat kirish uchun ishlatiladi (aslida GPT-2 ham tie qiladi,
lekin masshtab boshqacha). Qwen'da esa **bitta matritsa ikki vazifani** bajaradi:

1. **Kirish:** `embed_tokens(ids)` → vektor
2. **Chiqish:** `logits = hidden @ embed_tokens.weightᵀ` → 151936 ta logit

Logitlar juda katta bo'lib ketmasligi uchun (softmax to'yinib qolmasligi uchun)
embedding og'irliklari **kichik amplitudada** saqlanadi. Bu — arxitekturaviy
majburiyatning bevosita natijasi.

**Amaliy oqibat:** Qwen'ning **xom** (raw) embeddinglarini to'g'ridan-to'g'ri
semantik o'xshashlik uchun ishlatish GPT-2 ga qaraganda ham yomonroq natija
beradi. Semantika birinchi navbatda **kontekstual** hidden state'larda paydo bo'ladi.
Buni CELL 30 da isbotlaymiz.


## 6. Tokenizatsiya: 50 257 vs 151 936

Dars kodi:
```python
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "I love AI"
ids = tokenizer.encode(text)   # [40, 1842, 9552]


In [17]:
tests = [
    "I love AI",
    "Machine learning is transforming the world.",
    "Men sun'iy intellektni o'rganyapman.",
    "Salom dunyo! Bu o'zbek tilidagi jumla.",
    "人工智能",
    "def compute_attention(q, k, v):",
]

for text in tests:
    g_ids = gpt2_tok.encode(text)
    q_ids = qwen_tok.encode(text)
    print(f' "{text}"')
    print(f"   GPT-2 : {len(g_ids):>2} token  {g_ids}")
    print(f"           {gpt2_tok.convert_ids_to_tokens(g_ids)}")
    print(f"   Qwen  : {len(q_ids):>2} token  {q_ids}")
    print(f"           {qwen_tok.convert_ids_to_tokens(q_ids)}")
    ratio = len(g_ids) / max(len(q_ids), 1)
    print(f"   → Qwen {ratio:.2f}x samaraliroq\n")

 "I love AI"
   GPT-2 :  3 token  [40, 1842, 9552]
           ['I', 'Ġlove', 'ĠAI']
   Qwen  :  3 token  [40, 2948, 15235]
           ['I', 'Ġlove', 'ĠAI']
   → Qwen 1.00x samaraliroq

 "Machine learning is transforming the world."
   GPT-2 :  7 token  [37573, 4673, 318, 25449, 262, 995, 13]
           ['Machine', 'Ġlearning', 'Ġis', 'Ġtransforming', 'Ġthe', 'Ġworld', '.']
   Qwen  :  7 token  [21605, 6832, 374, 45790, 279, 1879, 13]
           ['Machine', 'Ġlearning', 'Ġis', 'Ġtransforming', 'Ġthe', 'Ġworld', '.']
   → Qwen 1.00x samaraliroq

 "Men sun'iy intellektni o'rganyapman."
   GPT-2 : 16 token  [10418, 4252, 6, 7745, 493, 13485, 21841, 8461, 267, 6, 81, 1030, 88, 499, 805, 13]
           ['Men', 'Ġsun', "'", 'iy', 'Ġint', 'elle', 'kt', 'ni', 'Ġo', "'", 'r', 'gan', 'y', 'ap', 'man', '.']
   Qwen  : 16 token  [28719, 7015, 6, 16220, 526, 6712, 74, 1517, 72, 297, 6, 1984, 3767, 391, 1515, 13]
           ['Men', 'Ġsun', "'", 'iy', 'Ġint', 'elle', 'k', 'tn', 'i', 'Ġo', "'", 'rg', '

### Tahlil: nega bu muhim?

1. **Inglizcha matnda** farq deyarli yo'q (~1.0×) — ikkala BPE ham ingliz
   korpusida yaxshi o'qitilgan.

2. **O'zbekcha matnda** GPT-2 so'zlarni harflarga parchalab tashlaydi
   (`o'rganyapman` → `o`, `'`, `rgan`, `yap`, `man`...). Qwen esa
   ancha butun bo'laklarni saqlaydi.

3. **Xitoycha `人工智能`** — GPT-2 uni **bayt darajasida** (UTF-8 byte fallback)
   parchalaydi, 12 tagacha token. Qwen 2 tokenda ifodalaydi.

**Iqtisodiy oqibat:** token soni = (a) API narxi, (b) kontekst oynasi sarfi,
(c) inference vaqti. O'zbek tilida ishlaydigan tizim uchun GPT-2 lug'ati
**tanlash mumkin bo'lmagan** variant.

**Arxitekturaviy narx:** 151 936 × 896 = 136 M parametr faqat embedding uchun.
Bu modelning 27% i. Ko'p tillilik **bepul emas**.


In [19]:
def tok_report(tok, name):
    print(f"{name}  ({type(tok).__name__})")
    print(f"  eos : {tok.eos_token!r:<18} id={tok.eos_token_id}")
    print(f"  pad : {tok.pad_token!r:<18} id={tok.pad_token_id}")
    print(f"  bos : {tok.bos_token!r:<18} id={tok.bos_token_id}")
    print(f"  unk : {tok.unk_token!r:<18} id={tok.unk_token_id}")
    print(f"  is_fast : {getattr(tok, 'is_fast', 'n/a')}")
    print(f"  all_special_tokens : {tok.all_special_tokens}")
    print()

tok_report(gpt2_tok, "GPT-2")
tok_report(qwen_tok, "Qwen2.5")

# --- Qo'shilgan (added) tokenlar: barcha versiyalarda ishlaydi ---
added = getattr(qwen_tok, "added_tokens_decoder", {})
print(f"Qwen'ga QO'SHILGAN tokenlar soni: {len(added)}")
print("-"*52)
for tid in sorted(added)[:25]:
    t = added[tid]
    content = t.content if hasattr(t, "content") else str(t)
    print(f"  {tid:>7}  {content}")
if len(added) > 25:
    print(f"  ... yana {len(added)-25} ta")

print(f"\n  Real lug'at (len(tokenizer))  : {len(qwen_tok):,}")
print(f"  Config vocab_size             : {qwen.config.vocab_size:,}")
print(f"  Farq (reserved / padding)     : {qwen.config.vocab_size - len(qwen_tok):,}")

print(f"\n  GPT-2 uchun:")
print(f"  len(tokenizer)={len(gpt2_tok):,}  vs  config={gpt2.config.vocab_size:,}"
      f"  farq={gpt2.config.vocab_size - len(gpt2_tok)}")


GPT-2  (GPT2Tokenizer)
  eos : '<|endoftext|>'    id=50256
  pad : None               id=None
  bos : '<|endoftext|>'    id=50256
  unk : '<|endoftext|>'    id=50256
  is_fast : True
  all_special_tokens : ['<|endoftext|>']

Qwen2.5  (Qwen2Tokenizer)
  eos : '<|endoftext|>'    id=151643
  pad : '<|endoftext|>'    id=151643
  bos : None               id=None
  unk : None               id=None
  is_fast : True
  all_special_tokens : ['<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']

Qwen'ga QO'SHILGAN tokenlar soni: 22
----------------------------------------------------
   151643  <|endoftext|>
   151644  <|im_start|>
   151645  <|im_end|>
   151646  <|object_ref_start|>
   151647  <|object_ref_end|>
   151648  <|box_start|>
   151649  <|box_end|>
   151650  <|quad_start|>
   151651  <|quad

In [ ]:
def to_id_list(x):
    """apply_chat_template / tokenizer chiqishini har qanday versiyada
    oddiy list[int] ga keltiradi."""
    import torch as _t
    if isinstance(x, _t.Tensor):
        return x.reshape(-1).tolist()
    if hasattr(x, "input_ids"):            # BatchEncoding
        return to_id_list(x.input_ids)
    if hasattr(x, "ids"):                  # tokenizers.Encoding
        return list(x.ids)
    if isinstance(x, (list, tuple)):
        if len(x) and isinstance(x[0], (list, tuple)):
            return list(x[0])
        if len(x) and not isinstance(x[0], int):
            return to_id_list(x[0])
        return list(x)
    raise TypeError(type(x))


print("1) CHAT TEMPLATE MAVJUDMI?")
print(f"   GPT-2 : {getattr(gpt2_tok, 'chat_template', None) is not None}")
print(f"   Qwen  : {getattr(qwen_tok, 'chat_template', None) is not None}")

msgs = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Salom!"},
]

if getattr(qwen_tok, "chat_template", None):
    
    rendered = qwen_tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    print("\n   Render qilingan ko'rinish (raw):")
    print("   " + repr(rendered))

    print("\n   O'qish uchun qulay ko'rinish:")
    for line in rendered.split("\n"):
        print("      | " + line)

    
    ids = to_id_list(qwen_tok(rendered, add_special_tokens=False))

    print(f"\n   Jami {len(ids)} token. Dastlabki 15 tasi:")
    for i in ids[:15]:
        print(f"      {i:>7}  {qwen_tok.decode([i])!r}")

    
    im_start = qwen_tok.convert_tokens_to_ids("<|im_start|>")
    im_end   = qwen_tok.convert_tokens_to_ids("<|im_end|>")
    print(f"\n   <|im_start|> id={im_start}, matnda {ids.count(im_start)} marta")
    print(f"   <|im_end|>   id={im_end}, matnda {ids.count(im_end)} marta")

print("\n2) FIM (Fill-In-Middle) TOKENLARI")
for t in ["<|fim_prefix|>", "<|fim_middle|>", "<|fim_suffix|>",
          "<|fim_pad|>", "<|repo_name|>", "<|file_sep|>"]:
    tid = qwen_tok.convert_tokens_to_ids(t)
    status = "OK" if tid is not None and tid != qwen_tok.unk_token_id else "yo'q"
    print(f"   {t:<18} -> {tid}   [{status}]")

print("\n3) AGENT / TOOL TOKENLARI")
for t in ["<tool_call>", "</tool_call>", "<tool_response>", "</tool_response>"]:
    print(f"   {t:<18} -> {qwen_tok.convert_tokens_to_ids(t)}")

print("\n4) VISION TOKENLARI (Qwen2-VL uchun zaxira)")
for t in ["<|vision_start|>", "<|vision_end|>", "<|image_pad|>", "<|video_pad|>"]:
    print(f"   {t:<18} -> {qwen_tok.convert_tokens_to_ids(t)}")


1) CHAT TEMPLATE MAVJUDMI?
   GPT-2 : False
   Qwen  : True

   Render qilingan ko'rinish (raw):
   '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nSalom!<|im_end|>\n<|im_start|>assistant\n'

   O'qish uchun qulay ko'rinish:
      | <|im_start|>system
      | You are a helpful assistant.<|im_end|>
      | <|im_start|>user
      | Salom!<|im_end|>
      | <|im_start|>assistant
      | 

   Jami 22 token. Dastlabki 15 tasi:
       151644  '<|im_start|>'
         8948  'system'
          198  '\n'
         2610  'You'
          525  ' are'
          264  ' a'
        10950  ' helpful'
        17847  ' assistant'
           13  '.'
       151645  '<|im_end|>'
          198  '\n'
       151644  '<|im_start|>'
          872  'user'
          198  '\n'
        17245  'Sal'

   <|im_start|> id=151644, matnda 3 marta
   <|im_end|>   id=151645, matnda 2 marta

2) FIM (Fill-In-Middle) TOKENLARI
   <|fim_prefix|>     -> 151659   [OK]
   <|fim_middle|>     -> 151660  

### ChatML formati: tokenizer darajasidagi protokol

Render natijasi quyidagicha bo'ladi:




Bu **ChatML** (Chat Markup Language). Uch jihati diqqatga sazovor:

**1. Rol chegaralari matn emas, TOKEN**

GPT-2 davrida dialogni shunday yozish kerak edi:


Bu oddiy matn — model `"Human:"` ni oddiy so'z sifatida ko'radi va
foydalanuvchi o'z xabarida `"AI:"` deb yozsa, **prompt injection** sodir bo'ladi.

Qwen'da `<|im_start|>` — bu **151644-ID**, uni oddiy matndan yozib bo'lmaydi
(tokenizer uni maxsus token sifatida ajratib oladi). Ya'ni rol chegarasi
**arxitekturaviy himoyalangan**.

**2. `add_generation_prompt=True` ning roli**

Oxirida ochiq qolgan `<|im_start|>assistant\n` — bu modelga
"endi sening navbating" degan signal. Bu qatorni qo'ymasangiz, model
foydalanuvchi xabarini davom ettirishga urinishi mumkin.

**3. `chat_template` — Jinja2 shabloni, model fayli emas**

U `tokenizer_config.json` ichida saqlanadi. Ya'ni bu **arxitektura emas,
konvensiya**. Lekin model shu konvensiyada o'qitilgani uchun undan chetga
chiqish sifatni sezilarli tushiradi.

> ⚠️ **Muhim nuance:** biz `Qwen2.5-0.5B` (base) modelini ishlatyapmiz —
> u **instruction-tuned emas**. Chat template tokenizer'da mavjud, lekin
> model unga muvofiq javob berishga o'qitilmagan. Haqiqiy dialog uchun
> `Qwen/Qwen2.5-0.5B-Instruct` kerak. Biz esa arxitekturani o'rganayotganimiz
> uchun base variant to'g'ri tanlov.

### GPT-2 bilan taqqoslash

| | GPT-2 | Qwen2.5 |
|---|---|---|
| `chat_template` | `None` | Jinja2 shabloni bor |
| Rol chegaralari | yo'q (oddiy matn) | maxsus token (151644/151645) |
| Tool calling | yo'q | `<tool_call>` tokenlari |
| Kod to'ldirish | yo'q | FIM tokenlari |
| Prompt injection himoyasi | yo'q | tokenizer darajasida |

**Xulosa:** 2019 → 2024 o'zgarishi faqat transformer bloki ichida emas.
**Lug'atning o'zi** modelning qanday ishlatilishini oldindan belgilab beradi.
GPT-2 lug'ati "matnni davom ettir" deydi, Qwen lug'ati "dialog yurit,
funksiya chaqir, kodni to'ldir, rasm qabul qil" deydi.
